# Highland Square Conjoint — Notebook 1 (REBUILT)

## Key changes from the previous version

1. **Letter-randomized presentation** — not just profile shuffling, but also random assignment of which physical position gets the label A/B/C. This breaks residual LLM positional bias.
2. **Strengthened Option D framing** — explicit permission to refuse, with persona-anchored context. The LLM was avoiding D because it pattern-matched 'pick something helpful'.
3. **Sonnet 4.5 model** (`claude-sonnet-4-5`) — better persona embodiment than Haiku for nuanced budget/preference reasoning.
4. **Rate-limit-aware concurrency** — concurrency=4 with 1.5s pacing to stay under typical tier limits.
5. **B-bias validation harness** — before the full run, a diagnostic confirms letter randomization is working.

## Pipeline
1. Attributes & realism rules
2. Generate valid profile pool
3. Build balanced choice tasks (greedy-swap D-efficient design)
4. Personas (pure demographics)
5. Prompt formatter with D option + letter randomization
6. **B-bias diagnostic pilot** (40 calls — validates the fix)
7. Full run with Sonnet 4.5
8. Save raw responses

**Output:** `data/raw_responses.csv` ready for Notebook 2.

## 0. Setup

In [ ]:
# !pip install anthropic pandas numpy --quiet

In [ ]:
import os
import json
import random
import asyncio
import itertools
import time
import re
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import anthropic

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

os.environ["ANTHROPIC_API_KEY"] = "NA"
assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY env var first."
print("Setup complete.")

Setup complete.


## 1. Attributes & Levels (locked)

In [2]:
ATTRIBUTES = {
    "Size": {
        "750 SF (compact 1BR)": 750,
        "1,000 SF (large 1BR / compact 2BR)": 1000,
        "1,250 SF (standard 2BR)": 1250,
    },
    "Price": {
        "$1,650/mo": 1650,
        "$1,950/mo": 1950,
        "$2,250/mo": 2250,
        "$2,550/mo": 2550,
    },
    "MoveInSpecial": {
        "None": None,
        "1 month free (12-mo lease)": None,
        "2 months free (13-mo lease)": None,
    },
    "Location": {
        "North Druid Hills / Briarcliff": None,
        "Virginia-Highland / Morningside": None,
        "Decatur / Emory Village": None,
        "Buckhead / Lenox": None,
    },
    "CommuteToWork": {
        "Quick (under 15 min by car)": None,
        "Average (15-30 min by car)": None,
        "Long (over 30 min by car)": None,
    },
    "Walkability": {
        "Car-Required (drive for groceries, dining, errands from this building)": None,
        "Walkable Errands (groceries & a few restaurants within a 10-min walk of this building)": None,
        "Walk Everywhere (daily errands, dining, transit within a 10-min walk of this building)": None,
    },
    "Finishes": {
        "Builder-grade (laminate counters, basic appliances, no smart features)": None,
        "Mid-tier (granite/quartz counters, stainless appliances, in-unit washer/dryer)": None,
        "Premium (quartz waterfall island, smart thermostat, keyless entry, video doorbell)": None,
    },
    "Parking": {
        "Surface lot, unassigned parking, no gate": None,
        "Gated surface lot + reserved space option": None,
        "Dedicated garage with assigned space + EV charging": None,
    },
    "Security": {
        "Tier 1: Open access, no controlled entry": None,
        "Tier 2: Perimeter gate + controlled-access lobby + camera coverage": None,
        "Tier 3: Tier 2 + 24/7 staff or virtual concierge + smart locks throughout": None,
    },
    "Rooftop": {
        "No rooftop space": None,
        "Rooftop lounge with skyline views & outdoor seating": None,
    },
    "Coworking": {
        "No dedicated coworking space": None,
        "Resident co-working lounge with private call rooms & wifi": None,
    },
    "PetAmenities": {
        "No pet amenities": None,
        "Standard dog park only": None,
        "Dog park + pet spa with grooming station": None,
    },
    "PackageHandling": {
        "Standard mailroom (sign for packages during office hours)": None,
        "24/7 Amazon Hub lockers + refrigerated grocery locker": None,
    },
}

attr_names = list(ATTRIBUTES.keys())
level_lists = [list(ATTRIBUTES[a].keys()) for a in attr_names]
total_full_factorial = int(np.prod([len(v) for v in ATTRIBUTES.values()]))
print(f"Total attributes: {len(ATTRIBUTES)}")
print(f"Full factorial profile space: {total_full_factorial:,} profiles")

Total attributes: 13
Full factorial profile space: 839,808 profiles


## 2. Realism Rules

In [3]:
def is_valid_profile(profile: dict) -> bool:
    sec = profile["Security"]
    fin = profile["Finishes"]
    park = profile["Parking"]
    rooftop = profile["Rooftop"]
    sf = ATTRIBUTES["Size"][profile["Size"]]
    price = ATTRIBUTES["Price"][profile["Price"]]

    # Rule 1: Price/SF realism band
    psf = price / sf
    if psf < 1.40 or psf > 3.20:
        return False

    # Rule 2: Premium smart finishes imply at least Tier 2 security
    if fin.startswith("Premium") and sec.startswith("Tier 1"):
        return False

    # Rule 3: Tier 3 security implies at least gated parking
    if sec.startswith("Tier 3") and park.startswith("Surface lot, unassigned"):
        return False

    # Rule 4: Rooftop lounge implies at least Tier 2 security
    if rooftop.startswith("Rooftop lounge") and sec.startswith("Tier 1"):
        return False

    return True

all_profiles = []
for combo in itertools.product(*level_lists):
    profile = dict(zip(attr_names, combo))
    if is_valid_profile(profile):
        all_profiles.append(profile)

print(f"Full factorial:    {total_full_factorial:,}")
print(f"Valid (realistic): {len(all_profiles):,} ({100*len(all_profiles)/total_full_factorial:.1f}% retained)")

Full factorial:    839,808
Valid (realistic): 466,560 (55.6% retained)


## 3. Build Balanced Choice Tasks (Greedy-Swap D-Efficient)

In [4]:
N_TASKS = 36
N_ALTS = 3
N_RANDOM_STARTS = 50
N_SWAPS_PER_START = 2000

def build_random_design(pool, n_tasks, n_alts, rng):
    return [rng.sample(pool, n_alts) for _ in range(n_tasks)]

def level_balance_score(design, attr_names, level_lists):
    flat = [alt for task in design for alt in task]
    s = 0
    for i, attr in enumerate(attr_names):
        target = len(flat) / len(level_lists[i])
        obs = {l: 0 for l in level_lists[i]}
        for p in flat:
            obs[p[attr]] += 1
        s += sum((c - target) ** 2 for c in obs.values())
    return s

def greedy_swap_optimize(design, pool, attr_names, level_lists, n_swaps, rng):
    current = level_balance_score(design, attr_names, level_lists)
    n_tasks = len(design)
    n_alts = len(design[0])
    for _ in range(n_swaps):
        task_idx = rng.randrange(n_tasks)
        alt_idx = rng.randrange(n_alts)
        candidate = rng.choice(pool)
        if any(candidate == design[task_idx][k] for k in range(n_alts) if k != alt_idx):
            continue
        original = design[task_idx][alt_idx]
        design[task_idx][alt_idx] = candidate
        new_score = level_balance_score(design, attr_names, level_lists)
        if new_score < current:
            current = new_score
        else:
            design[task_idx][alt_idx] = original
    return design, current

rng = random.Random(RANDOM_SEED)
best_design = None
best_score = float("inf")
for start in range(N_RANDOM_STARTS):
    cand = build_random_design(all_profiles, N_TASKS, N_ALTS, rng)
    cand, s = greedy_swap_optimize(cand, all_profiles, attr_names, level_lists, N_SWAPS_PER_START, rng)
    if s < best_score:
        best_score = s
        best_design = cand
    if (start + 1) % 10 == 0:
        print(f"  Restart {start + 1}/{N_RANDOM_STARTS} | Best score: {best_score:.1f}")

print(f"\nFinal best level-balance score: {best_score:.1f}")
print(f"Total alternatives in design: {N_TASKS * N_ALTS}")

  Restart 10/50 | Best score: 10.0
  Restart 20/50 | Best score: 10.0
  Restart 30/50 | Best score: 10.0
  Restart 40/50 | Best score: 10.0
  Restart 50/50 | Best score: 10.0

Final best level-balance score: 10.0
Total alternatives in design: 108


In [5]:
# Save design
design_records = []
for task_id, task in enumerate(best_design):
    for alt_id, profile in enumerate(task):
        rec = {"Task_ID": task_id, "Alt_ID": alt_id}
        rec.update(profile)
        design_records.append(rec)
pd.DataFrame(design_records).to_csv(DATA_DIR / "choice_tasks.csv", index=False)
print(f"Saved {DATA_DIR / 'choice_tasks.csv'}")

Saved data/choice_tasks.csv


## 4. Personas (pure demographics)

In [6]:
PERSONAS = {
    "emory_grad": """You are Maya, 28. You are a third-year PhD candidate.
Your stipend is $42,000/year plus $10,000/year in research assistantship funding. You have $580/month in federal student loan payments. You have $4,200 in liquid savings and no other significant assets.
You moved to Atlanta from Wisconsin two years ago. You expect to live in this apartment for at least 18 months, through the end of your dissertation.""",

    "young_professional": """You are David, 34. You are a senior consultant at a healthcare strategy firm.
Your base salary is $135,000/year with a typical annual bonus of $20,000. You have $42,000 in a high-yield savings account and a 401(k) balance of $100,000. You have no debt other than student loans paid down to $14,000. You moved from Boston eight months ago.
Your daily destination is your firm's office three days a week; you work from home the other two days. You expect to live in this apartment for 2-3 years.""",

    "empty_nester": """You are Patricia, 58. You recently retired from a 25-year career as a hospital administrator. Your husband Tom, 60, still works as a CPA.
Combined household income is $180,000/year. You sold your 4-bedroom Sandy Springs home last September for $815,000 (purchased in 1998 for $310,000). You and Tom have approximately $1.4M in retirement and brokerage accounts.
You and Tom are renting for the first time in 30 years. You have no children at home - your two adult children live in Charlotte and Denver.
You expect to live in this apartment for 2-4 years while deciding whether to purchase a smaller condo.""",

    "skeptical_renter_control": """You are Alex, 31. You are a software engineer at a mid-sized fintech company in Atlanta.
Your salary is $115,000/year. You have $35,000 in liquid savings, a 401(k) balance of $95,000, and no significant debt. You have lived in Atlanta for four years. You expect to live in this apartment for 1-2 years.""",
}

for name, prompt in PERSONAS.items():
    print(f"=== {name} ===")
    print(prompt[:200] + "...\n")

=== emory_grad ===
You are Maya, 28. You are a third-year PhD candidate.
Your stipend is $42,000/year plus $10,000/year in research assistantship funding. You have $580/month in federal student loan payments. You have $...

=== young_professional ===
You are David, 34. You are a senior consultant at a healthcare strategy firm.
Your base salary is $135,000/year with a typical annual bonus of $20,000. You have $42,000 in a high-yield savings account...

=== empty_nester ===
You are Patricia, 58. You recently retired from a 25-year career as a hospital administrator. Your husband Tom, 60, still works as a CPA.
Combined household income is $180,000/year. You sold your 4-be...

=== skeptical_renter_control ===
You are Alex, 31. You are a software engineer at a mid-sized fintech company in Atlanta.
Your salary is $115,000/year. You have $35,000 in liquid savings, a 401(k) balance of $95,000, and no significa...



## 5. Prompt Formatter with Letter Randomization + Strengthened D Option

**Two key fixes here:**

1. **Letter randomization.** Each call independently shuffles which of the 3 profiles gets called "A", "B", or "C". This breaks any residual LLM bias toward position B because B no longer corresponds to any fixed profile slot — it's randomly which profile.

2. **Strengthened D option.** We make refusing explicit and legitimate, not a fallback. The framing tells the LLM that picking D when none fits is the RIGHT answer for a careful shopper, not a failure to choose.

The function returns both the prompt and the letter-to-profile mapping so we can reconstruct which profile was actually chosen.

In [7]:
def format_choice_prompt(task_profiles, attr_names, rng):
    """
    Format a single choice task with randomized letter assignment.
    
    Returns:
        prompt: the user message to send Claude
        letter_to_profile: dict mapping A/B/C -> profile dict (so we can reconstruct
            which actual profile the response refers to, regardless of order)
    """
    # Step 1: randomly shuffle which profile gets which letter
    profiles_copy = list(task_profiles)
    rng.shuffle(profiles_copy)
    letter_to_profile = {chr(ord('A') + j): p for j, p in enumerate(profiles_copy)}
    
    # Step 2: build the prompt
    lines = [
        "You are evaluating three apartment options in Atlanta. Please read all three carefully.\n",
        "Note: 'Location' is the broader submarket. 'Walkability' describes the immediate block "
        "around the building (every Atlanta submarket has both walkable enclaves and "
        "car-dependent stretches, so the two attributes can vary independently). "
        "Use only the information provided. Do not infer additional facts about commute times, "
        "neighborhood character, building age, or features not listed.\n",
    ]
    for letter, profile in letter_to_profile.items():
        lines.append(f"Option {letter}:")
        for attr in attr_names:
            lines.append(f"  - {attr}: {profile[attr]}")
        lines.append("")
    
    # Strengthened D option — make refusing legitimate, not a fallback
    lines.append("Option D: None of the above are right for you.")
    lines.append("  - Choose this if NO option fits your budget, your situation, or your needs.")
    lines.append("  - Picking D is the correct answer when nothing genuinely fits — a careful shopper walks away from a bad fit rather than settling.")
    lines.append("")
    lines.append(
        "Decision: Reply with just the letter (A, B, C, or D) on its own line, "
        "followed by one short sentence explaining your reasoning grounded in your specific situation (your income, your savings, your commute, your timeline)."
    )
    return "\n".join(lines), letter_to_profile

In [8]:
# Show one example prompt so you can sanity-check it
rng_demo = random.Random(123)
demo_prompt, demo_map = format_choice_prompt(best_design[0], attr_names, rng_demo)
print(demo_prompt)
print("\n--- Letter mapping for this example ---")
for letter, prof in demo_map.items():
    print(f"  {letter}: Price={prof['Price']}, Location={prof['Location']}")

You are evaluating three apartment options in Atlanta. Please read all three carefully.

Note: 'Location' is the broader submarket. 'Walkability' describes the immediate block around the building (every Atlanta submarket has both walkable enclaves and car-dependent stretches, so the two attributes can vary independently). Use only the information provided. Do not infer additional facts about commute times, neighborhood character, building age, or features not listed.

Option A:
  - Size: 1,000 SF (large 1BR / compact 2BR)
  - Price: $2,550/mo
  - MoveInSpecial: 1 month free (12-mo lease)
  - Location: Decatur / Emory Village
  - CommuteToWork: Long (over 30 min by car)
  - Walkability: Walk Everywhere (daily errands, dining, transit within a 10-min walk of this building)
  - Finishes: Mid-tier (granite/quartz counters, stainless appliances, in-unit washer/dryer)
  - Parking: Surface lot, unassigned parking, no gate
  - Security: Tier 2: Perimeter gate + controlled-access lobby + camera

## 6. B-Bias Diagnostic Pilot

Run 40 calls before the full experiment to verify two things:

1. **Letter randomization is breaking the B-bias.** With profiles randomly assigned to letters, no specific profile should systematically land in position B. The choice distribution across A/B/C should be roughly equal (~30-37% each, given the D option pulls some out).
2. **D is being used at least occasionally.** Maya facing a card with all $2,250+ options should sometimes pick D. If D is at 0% across all personas, the strengthening didn't work and we need stronger language.

**Cost:** 40 API calls, ~2 minutes.

In [11]:
MODEL = "claude-sonnet-4-5"  # upgraded from haiku for better persona embodiment
DIAGNOSTIC_TASKS = best_design[:10]    # 10 different tasks
DIAGNOSTIC_REPS = 1                    # 1 rep per persona per task = 10 × 4 × 1 = 40 calls

client = anthropic.Anthropic()
diag_results = []
rng_diag = random.Random(RANDOM_SEED)

print(f"Running diagnostic: {len(DIAGNOSTIC_TASKS) * len(PERSONAS) * DIAGNOSTIC_REPS} calls...\n")

for task_id, task in enumerate(DIAGNOSTIC_TASKS):
    for persona_name in PERSONAS:
        for rep in range(DIAGNOSTIC_REPS):
            prompt, letter_map = format_choice_prompt(task, attr_names, rng_diag)
            try:
                response = client.messages.create(
                    model=MODEL,
                    max_tokens=200,
                    temperature=1.0,
                    system=PERSONAS[persona_name],
                    messages=[{"role": "user", "content": prompt}],
                )
                text = response.content[0].text
                diag_results.append({
                    "Task_ID": task_id,
                    "Persona": persona_name,
                    "Rep": rep,
                    "Letter_Map": json.dumps({k: v for k, v in letter_map.items()}),
                    "Response": text,
                })
                # Quick parse for inline display
                m = re.search(r'\b([ABCD])\b', text[:50])
                letter = m.group(1) if m else '?'
                print(f"T{task_id} {persona_name:25s} → {letter} | {text[:70]}")
                time.sleep(1.5)  # pace to respect rate limits
            except Exception as e:
                print(f"Error: {e}")
                time.sleep(5)

print(f"\nDiagnostic complete: {len(diag_results)} responses")

Running diagnostic: 40 calls...

Error: Request timed out or interrupted. This could be due to a network timeout, dropped connection, or request cancellation. See https://docs.anthropic.com/en/api/errors#long-requests for more details.
Error: Request timed out or interrupted. This could be due to a network timeout, dropped connection, or request cancellation. See https://docs.anthropic.com/en/api/errors#long-requests for more details.
T0 empty_nester              → B | B

With Tom still working as a CPA, the long commute is irrelevant sin
T0 skeptical_renter_control  → B | B

The walkability and coworking space are ideal for my software engin
T1 emory_grad                → B | B

Option B offers the best balance for a PhD candidate: the quick com
T1 young_professional        → A | A

Option A offers the best balance of a reasonable commute (critical 
T1 empty_nester              → C | C

The premium finishes and short commute for Tom matter most during o
T1 skeptical_renter_control  → 

In [12]:
# Analyze the diagnostic
diag_df = pd.DataFrame(diag_results)
diag_df['Choice_Letter'] = diag_df['Response'].apply(
    lambda t: re.search(r'\b([ABCD])\b', str(t)[:50]).group(1)
    if re.search(r'\b([ABCD])\b', str(t)[:50]) else None
)

print("=== Choice distribution across all 40 diagnostic calls ===")
print(diag_df['Choice_Letter'].value_counts())
print()
print("=== Per-persona choice distribution ===")
print(pd.crosstab(diag_df['Persona'], diag_df['Choice_Letter'], normalize='index').round(3) * 100)
print()
print("INTERPRETATION:")
print("  - A/B/C should be roughly equal (~30-35% each) if letter randomization works")
print("  - D should appear at least occasionally (especially for emory_grad on expensive cards)")
print("  - If B is still >40% across the board, the bias is in the LLM and we need to investigate further")

=== Choice distribution across all 40 diagnostic calls ===
Choice_Letter
B    22
C    13
A     3
Name: count, dtype: int64

=== Per-persona choice distribution ===
Choice_Letter                A     B     C
Persona                                   
emory_grad                 0.0  66.7  33.3
empty_nester               0.0  50.0  50.0
skeptical_renter_control   0.0  50.0  50.0
young_professional        33.3  66.7   0.0

INTERPRETATION:
  - A/B/C should be roughly equal (~30-35% each) if letter randomization works
  - D should appear at least occasionally (especially for emory_grad on expensive cards)
  - If B is still >40% across the board, the bias is in the LLM and we need to investigate further


In [13]:
# For each response, reconstruct which actual profile was chosen
import json

diag_df['Letter_Map_Parsed'] = diag_df['Letter_Map'].apply(json.loads)

def get_chosen_profile(row):
    if not row['Choice_Letter']:
        return None
    return row['Letter_Map_Parsed'].get(row['Choice_Letter'])

diag_df['Chosen_Profile'] = diag_df.apply(get_chosen_profile, axis=1)

# Check: when the LLM picks "B", which profile (by price) is it?
b_picks = diag_df[diag_df['Choice_Letter'] == 'B']
b_picks['Chosen_Price'] = b_picks['Chosen_Profile'].apply(lambda p: p['Price'] if p else None)
print("When LLM picks 'B', what is the price of the chosen profile?")
print(b_picks['Chosen_Price'].value_counts())

# Compare to: when LLM picks "A" or "C"
print("\nWhen LLM picks 'A':")
a_picks = diag_df[diag_df['Choice_Letter'] == 'A']
a_picks['Chosen_Price'] = a_picks['Chosen_Profile'].apply(lambda p: p['Price'] if p else None)
print(a_picks['Chosen_Price'].value_counts())

print("\nWhen LLM picks 'C':")
c_picks = diag_df[diag_df['Choice_Letter'] == 'C']
c_picks['Chosen_Price'] = c_picks['Chosen_Profile'].apply(lambda p: p['Price'] if p else None)
print(c_picks['Chosen_Price'].value_counts())

When LLM picks 'B', what is the price of the chosen profile?
Chosen_Price
$1,950/mo    8
$2,550/mo    6
$1,650/mo    6
$2,250/mo    2
Name: count, dtype: int64

When LLM picks 'A':
Chosen_Price
$2,550/mo    2
$1,650/mo    1
Name: count, dtype: int64

When LLM picks 'C':
Chosen_Price
$2,250/mo    6
$1,650/mo    4
$2,550/mo    2
$1,950/mo    1
Name: count, dtype: int64


/var/folders/5t/k32b1zys181cv4n2nz_7rq1c0000gp/T/ipykernel_3393/1016449236.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  b_picks['Chosen_Price'] = b_picks['Chosen_Profile'].apply(lambda p: p['Price'] if p else None)
/var/folders/5t/k32b1zys181cv4n2nz_7rq1c0000gp/T/ipykernel_3393/1016449236.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  a_picks['Chosen_Price'] = a_picks['Chosen_Profile'].apply(lambda p: p['Price'] if p else None)
/var/folders/5t/k32b1zys181cv4n2nz_7rq1c0000gp/T/ipykernel_3393/10

## STOP — Check the diagnostic before proceeding

**Pass criteria:**
- A, B, C distributed within ~5pp of each other (e.g., 28/32/30, not 22/45/28)
- D appears at least 1 time across the 40 calls
- Maya (emory_grad) ideally has the highest D-rate

**If the diagnostic passes**, run the full experiment cells below.

**If B is still dominant**, stop and reach out — we'll need to investigate whether the bias is at the model layer.

**If D is at 0%**, the personas need stronger budget-constraint framing, or we strengthen the D wording further.

## 7. Full Experiment (Sonnet 4.5, rate-limit-aware)

**Settings:**
- 36 tasks × 4 personas × 40 reps = **5,760 calls**
- Concurrency = 4 (well below Sonnet's 50 RPM tier-1 limit)
- Per-call pacing = 1.0s (so effective throughput ≈ 4 calls/sec ≈ 240/min, but the semaphore-throttling and natural API response times bring this comfortably under 50 RPM)
- Wall time: ~25-30 minutes
- Incremental checkpointing every 200 calls

In [ ]:
FULL_MODEL = "claude-sonnet-4-5"
REPETITIONS_PER_PERSONA = 40
CONCURRENCY = 4
PER_CALL_DELAY_SEC = 1.0  # extra pacing inside each async worker

total_calls = N_TASKS * len(PERSONAS) * REPETITIONS_PER_PERSONA
print(f"Full run: {total_calls:,} API calls.")
print(f"Concurrency={CONCURRENCY}, est wall time: ~{total_calls * (PER_CALL_DELAY_SEC + 2.5) / CONCURRENCY / 60:.0f} minutes")

In [ ]:
async def run_one_call(client, semaphore, task_id, task, persona_name, rep, attr_names, rng_seed):
    rng_local = random.Random(rng_seed)
    prompt, letter_map = format_choice_prompt(task, attr_names, rng_local)

    async with semaphore:
        for attempt in range(5):
            try:
                response = await client.messages.create(
                    model=FULL_MODEL,
                    max_tokens=200,
                    temperature=1.0,
                    system=PERSONAS[persona_name],
                    messages=[{"role": "user", "content": prompt}],
                )
                await asyncio.sleep(PER_CALL_DELAY_SEC)
                return {
                    "Task_ID": task_id,
                    "Persona": persona_name,
                    "Rep": rep,
                    "Letter_Map": json.dumps(letter_map),
                    "Response": response.content[0].text,
                    "Error": None,
                }
            except Exception as e:
                if attempt == 4:
                    return {
                        "Task_ID": task_id,
                        "Persona": persona_name,
                        "Rep": rep,
                        "Letter_Map": json.dumps(letter_map),
                        "Response": None,
                        "Error": str(e),
                    }
                # Aggressive backoff for rate limits: 5s, 10s, 20s, 40s
                await asyncio.sleep(5 * (2 ** attempt))
        return None

async def run_full_experiment():
    async_client = anthropic.AsyncAnthropic()
    semaphore = asyncio.Semaphore(CONCURRENCY)

    coros = []
    call_idx = 0
    for task_id, task in enumerate(best_design):
        for persona_name in PERSONAS:
            for rep in range(REPETITIONS_PER_PERSONA):
                seed = RANDOM_SEED + call_idx
                coros.append(run_one_call(async_client, semaphore, task_id, task,
                                          persona_name, rep, attr_names, seed))
                call_idx += 1

    print(f"Dispatching {len(coros)} calls at concurrency={CONCURRENCY}...")
    start = time.time()
    results = []
    chunk = 200
    for i in range(0, len(coros), chunk):
        batch = coros[i:i+chunk]
        batch_results = await asyncio.gather(*batch)
        results.extend(batch_results)
        elapsed = time.time() - start
        n_err = sum(1 for r in results if r and r.get('Error') is not None)
        # Incremental save
        pd.DataFrame(results).to_csv(DATA_DIR / "raw_responses_partial.csv", index=False)
        print(f"  {len(results)}/{len(coros)} done | errors so far: {n_err} | elapsed {elapsed:.0f}s — checkpoint saved")
    return results

all_results = await run_full_experiment()
print(f"\nDone. {len(all_results)} responses collected.")

In [ ]:
# Save final results
results_df = pd.DataFrame(all_results)
n_err = results_df['Error'].notna().sum()
print(f"Errors: {n_err} / {len(results_df)} ({100*n_err/len(results_df):.1f}%)")

out_path = DATA_DIR / "raw_responses.csv"
results_df.to_csv(out_path, index=False)
print(f"Saved: {out_path.resolve()}")

# Quick distribution check
good = results_df[results_df['Error'].isna()].copy()
good['Choice_Letter'] = good['Response'].apply(
    lambda t: re.search(r'\b([ABCD])\b', str(t)[:50]).group(1)
    if re.search(r'\b([ABCD])\b', str(t)[:50]) else None
)
print("\n=== Choice distribution (full run) ===")
print(good['Choice_Letter'].value_counts())
print("\n=== Per-persona ===")
print(pd.crosstab(good['Persona'], good['Choice_Letter'], normalize='index').round(3) * 100)